# 01 — Data and universe

Start here: the reviewed 95-market universe, its metadata, and the single Tushare-to-simulation pipeline that maintains it.

## Universe and metadata

This is the first notebook in a series about researching **Chinese futures**
with [pysystemtrade](https://github.com/robcarver17/pysystemtrade), using
daily data for every contract ever listed on the six mainland exchanges
(SHFE, DCE, CZCE, CFFEX, INE, GFEX), sourced from Tushare.

**What you will learn here**: how the repo models futures data, why all 95
reviewed histories belong in the research universe, how trailing volume makes
availability point-in-time, and where every piece of metadata lives.

## The data model in one picture

pysystemtrade never trades a "continuous future" — it stores *individual
contracts* and stitches them itself:

```
per-contract prices  ──roll calendar──▶  multiple prices  ──Panama stitch──▶  adjusted prices
(parquet, OHLC+FINAL+VOLUME)             (PRICE/CARRY/FORWARD                (single back-adjusted
 one file per contract                    + their contract ids)               series per instrument)
```

Storage split (all locations configured in `private/private_config.yaml`):

| What | Where |
|---|---|
| per-contract, multiple, adjusted prices; CNHUSD | parquet |
| contract expiries & sampling state | MongoDB |
| instrument metadata, roll parameters, spread costs, roll calendars | CSV in `data/futures/` |

Deeper reading: `docs/tushare_chinese_futures.md` (design),
`docs/tushare_data_inspection.md` (inspecting each stage),
`docs/backtesting.md` (the full simulation manual).

## Loading the simulation data

`dbFuturesSimData` is the database-backed simulation data object. The helper
below cross-checks its adjusted-price instruments against the stitchable
Tushare manifest. Present-day labels such as "dead" or "predecessor" do not
remove an instrument from its own earlier history.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import research as R

R.set_notebook_style()

In [ ]:
from sysdata.sim.db_futures_sim_data import dbFuturesSimData

data = dbFuturesSimData()
instruments = R.chinese_universe(data)
assert len(instruments) == 95
print(f"{len(instruments)} reviewed instruments with stitched histories")
print(instruments[:10], "...")

## Instrument metadata

Static metadata (point size, currency, asset class...) lives in
`data/futures/csvconfig/instrumentconfig.csv`; roll behaviour in
`rollconfig.csv`; half-spreads in `spreadcosts.csv` (imported into Mongo for
sim use). Naming is `EXCHANGE_CODE` — `SHFE_RB` is Shanghai rebar,
`CFFEX_IF` the CSI300 index future.

All instruments are quoted in CNY; the config says **CNH** so that the single
`CNHUSD` FX series can convert P&L for USD-based accounts. In this series we
run with `base_currency: "CNH"`, so FX is exactly 1 and pre-2012 history
(before the CNHUSD series starts) needs no FX at all.

In [ ]:
meta = data.get_all_instrument_data_as_df().loc[instruments].copy()
meta["Exchange"] = [code.split("_")[0] for code in meta.index]
meta["SpreadCost"] = [data.get_spread_cost(code) for code in meta.index]

from sysdata.csv.csv_roll_parameters import csvRollParametersData

rolls = csvRollParametersData().get_roll_parameters_all_instruments()
meta = meta.join(
    rolls[["HoldRollCycle", "RollOffsetDays", "CarryOffset", "PricedRollCycle"]]
)
meta[["Description", "Pointsize", "Currency", "AssetClass", "Exchange",
      "SpreadCost", "HoldRollCycle", "RollOffsetDays", "CarryOffset"]].head(12)

One metadata spot check is worth keeping visible. Plywood (`DCE_BB`) has a
perfectly finite configured half-spread, but that static cost object does not
make its later zombie market liquid. Price, cost, and point-in-time liquidity
are separate facts; the volume test below is what controls eligibility.

In [ ]:
bb_cost = data.get_raw_cost_data("DCE_BB")
print("DCE_BB static cost metadata:", bb_cost)

## The vendor manifest: renamed products and specification eras

Chinese exchanges have renamed or re-specified several products. Each era is
its **own instrument**, linked by a `Predecessor` column in the Tushare
manifest — so a backtest never accidentally splices two different contract
specs together. The famous case is DCE fibreboard, which changed its trading
unit in December 2019: one vendor contract crossing that date is split
between `DCE_FB_OLD` and `DCE_FB`.

In [ ]:
manifest = pd.read_csv(
    R.REPO_ROOT / "sysdata/tushare/config/futures_instruments.csv",
    keep_default_na=False,
)
renames = manifest[manifest["Predecessor"] != ""][
    ["Instrument", "Predecessor", "ValidFrom"]
]
print("Product renames / re-specifications (successor <- predecessor):")
renames

In [ ]:
manifest[manifest["FutCode"] == "FB"]  # the fibreboard era split

## What stitched histories exist?

The first and last adjusted-price rows describe the stored history; they are
not, by themselves, a liquidity test. A non-zero print can exist in a market
too thin to trade realistically. We keep present-day predecessor/dead labels
only for reporting and never feed them into historical eligibility.

In [ ]:
spans = {}
for code in instruments:
    prices = data.daily_prices(code)
    spans[code] = dict(first=prices.index[0], last=prices.index[-1],
                       days=len(prices))
spans = pd.DataFrame(spans).T
spans["first"] = pd.to_datetime(spans["first"])
spans["last"] = pd.to_datetime(spans["last"])
spans["years"] = ((spans["last"] - spans["first"]).dt.days / 365.25).round(1)
spans["present_day_label"] = "open history"
spans.loc[R.predecessor_instruments(), "present_day_label"] = "predecessor era"
spans.loc[R.KNOWN_DEAD_MARKETS, "present_day_label"] = "known dead market"

universe = meta.join(spans)
print(universe["present_day_label"].value_counts().to_string())
history_display = universe[[
    "AssetClass", "Exchange", "Pointsize", "SpreadCost",
    "first", "last", "years", "present_day_label",
]].sort_values("first")
history_display.head(15)

The terminal group has two flavours: a reviewed product rename/specification
boundary, or a market now known to have died. Calling either one "terminal"
uses information from the completed history. A backtest must instead react to
volume observed up to each date, with any final forced close disclosed as an
assumption.

In [ ]:
terminal_history = universe.loc[R.terminal_instruments(), [
    "AssetClass", "first", "last", "years", "present_day_label",
]].sort_values("last")
terminal_history

## Point-in-time liquidity: when could the backtest actually include one?

For each observed held-price row we read the raw volume of the contract that
supplied `PRICE`. A missing raw volume on such a row becomes zero (unknown is
unsafe); an exchange-closed date remains missing. The eligibility rule then
uses only the last 20 observed sessions:

- enter when mean held-contract volume reaches 130 lots;
- remain eligible until it falls below 70 lots;
- never backfill eligibility before the first qualifying date.

The gap between 130 and 70 is hysteresis: it prevents daily entry/exit churn
around one threshold. These are explicit research assumptions, not universal
claims that 130 lots makes every contract equally cheap to trade. A decision
uses volume known through that close; the native account machinery applies it
with its normal one-business-row fill delay.

In [ ]:
volumes = R.held_contract_volumes(data, instruments)
average_volume = R.trailing_liquidity(volumes)
eligibility = R.liquidity_eligibility(volumes, force_terminal_close=True)
events = R.liquidity_event_table(eligibility, volumes)

ever_eligible = set(eligibility.columns[eligibility.any(axis=0)])
never_eligible = set(instruments) - ever_eligible
forced_exit_instruments = set(events.loc[
    events["event"] == "forced terminal exit", "instrument"])
expected_never = {"CZCE_LR", "CZCE_PM"}
expected_forced = {
    "CZCE_ER", "CZCE_JR", "CZCE_ME", "CZCE_RO",
    "CZCE_RS", "CZCE_TC", "CZCE_WS", "CZCE_WT",
}
assert len(ever_eligible) == 93
assert int(eligibility.iloc[-1].sum()) == 78
assert never_eligible == expected_never
assert forced_exit_instruments == expected_forced

print(f"volume panel: {volumes.index[0].date()} to {volumes.index[-1].date()}, "
      f"{volumes.shape[1]} instruments")
print(f"ever eligible: {len(ever_eligible)}; never eligible: "
      f"{sorted(never_eligible)}")
print(f"eligible on final business date: {int(eligibility.iloc[-1].sum())}")
print(f"forced terminal closes: {sorted(forced_exit_instruments)}")
events.tail(10)

In [ ]:
eligible_by_exchange = eligibility.T.groupby(meta["Exchange"]).sum().T
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
eligibility.sum(axis=1).plot(
    ax=axes[0], title="Point-in-time eligible Chinese futures")
axes[0].set_ylabel("eligible instruments")
eligible_by_exchange.plot.area(
    ax=axes[1], alpha=0.75, linewidth=0,
    title="Eligible instruments by exchange")
axes[1].set_ylabel("eligible instruments")
axes[1].set_xlabel("decision date")
plt.tight_layout()

## Three liquidity histories, seen without hindsight

Rebar is a long-lived liquid market, plywood later dies, and GFEX lithium is a
recent listing. Plotting their trailing averages separately preserves each
market's scale. The two horizontal lines are the actual entry/exit decisions;
no future observation is used to move an earlier decision.

In [ ]:
case_studies = ["SHFE_RB", "DCE_BB", "GFEX_LC"]
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for axis, code_ in zip(axes, case_studies):
    average_volume[code_].plot(ax=axis, color="black")
    axis.axhline(R.LIQUIDITY_ENTRY, color="tab:green", linestyle="--",
                 label="entry threshold")
    axis.axhline(R.LIQUIDITY_EXIT, color="tab:red", linestyle=":",
                 label="exit threshold")
    axis.set_yscale("symlog", linthresh=1)
    axis.set_ylim(bottom=0)
    axis.set_title(code_)
    axis.set_ylabel("20-session mean lots (symlog)")
axes[0].legend()
axes[-1].set_xlabel("decision date")
plt.tight_layout()

## Terminal histories and the last executable exit

Normal liquidity exits are causal. A market can also stop printing while it
is still above the threshold; there is then no future quote on which a delayed
backtest can discover the disappearance. For the 16 reviewed predecessor/dead
histories, the helper explicitly sets the target to zero on the system
business row immediately before the final observed quote. That row can itself
be an exchange holiday, because native
`delayfill=True` shifts the target by one system row. The resulting fill uses
the real final quote and retains the complete reopening move.

That row is labelled `forced terminal exit`: it is a conservative liquidation
assumption, not information the strategy possessed at the time.

In [ ]:
terminal_events = events[
    events["instrument"].isin(R.terminal_instruments())
].sort_values(["instrument", "date"])
last_terminal_event = terminal_events.groupby("instrument").tail(1).set_index(
    "instrument")
terminal_report = terminal_history.join(last_terminal_event[[
    "date", "event", "mean_volume_20", "terminal_assumption",
]])
terminal_report.sort_values("last")

## Held price versus adjusted price: a DCE_JD spot check

`multiple.PRICE` is an actual close from the held contract. The adjusted
series adds historical roll gaps so that **price differences** remain usable
through rolls; its absolute level is synthetic and percentage changes of that
level are invalid. On every non-roll row, held and adjusted daily differences
must be identical.

In [ ]:
jd_multiple = pd.DataFrame(data.get_multiple_prices("DCE_JD"))
jd_adjusted = data.get_backadjusted_futures_price("DCE_JD")
jd_prices = pd.concat(
    {
        "held contract close": jd_multiple["PRICE"],
        "additive-Panama level": jd_adjusted,
    },
    axis=1,
    join="inner",
).dropna()

same_contract = jd_multiple["PRICE_CONTRACT"].eq(
    jd_multiple["PRICE_CONTRACT"].shift())
held_change = jd_multiple["PRICE"].diff()
adjusted_change = jd_adjusted.diff()
non_roll = same_contract & held_change.notna() & adjusted_change.notna()
max_non_roll_error = (held_change[non_roll] - adjusted_change[non_roll]).abs().max()
assert np.isclose(max_non_roll_error, 0.0)
print(f"maximum non-roll difference mismatch: {max_non_roll_error:.6f}")
display(jd_multiple.tail(5))

plot_start = "2022-01-01"
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
jd_prices["held contract close"].loc[plot_start:].plot(
    ax=axes[0], title="DCE_JD held-contract close")
axes[0].set_ylabel("actual price")
jd_prices["additive-Panama level"].loc[plot_start:].plot(
    ax=axes[1], title="DCE_JD additive-Panama level (absolute level arbitrary)")
axes[1].set_ylabel("adjusted price units")
axes[1].set_xlabel("date")
plt.tight_layout()

## The research universe used in the rest of the series

All 95 histories remain present. A successor does not erase its predecessor's
earlier opportunity, and a market known dead today is not removed from years
when its trailing volume passed the rule. Portfolio weights will be zero while
`eligibility` is false and are computed across the eligible set on each date.

This prevents literal pre-listing trades and avoids survivorship deletion.
The thresholds and terminal-close convention remain modelling assumptions and
will be carried visibly through later notebooks.

**Next**: the data-pipeline section below shows how those histories are
produced and updated.

In [ ]:
print("all histories retained:", len(instruments))
print("present-day predecessor labels:", R.predecessor_instruments())
print("present-day dead-market labels:", R.KNOWN_DEAD_MARKETS)

## Data pipeline and updating

The Tushare integration follows the repo's philosophy: Tushare is a *data
vendor* (like Barchart in the upstream docs), not a broker. It owns the
vendor-to-contract-price boundary; calendar, multiple-price, adjusted-price,
and live-roll logic stays in stock pysystemtrade.

| Task | Command |
|---|---|
| one-off full-history bootstrap (resumable) | `python -m sysinit.futures.seed_price_data_from_tushare` |
| **daily update** (contracts, prices, CNHUSD) | `python -m sysproduction.update_tushare_futures` |
| daily multiple + adjusted append | `python -m sysproduction.run_daily_update_multiple_adjusted_prices` |
| rebuild from accepted calendars | `python -m sysinit.futures.rebuild_tushare_multiple_adjusted --all` |

The daily update can also run under the production scheduler
(`run_daily_tushare_price_updates`, registered in
`syscontrol/control_config.yaml` and the crontab).

The final command is deliberately only a thin China-universe adapter around
the native single-instrument builders. It does not choose roll policy or
create calendars. Those are careful, manual initialization tasks taught in
notebook 02.

Full design: `docs/tushare_chinese_futures.md`. This notebook walks one
contract through every stage so you can *see* the pipeline.

## Stage 0 — the vendor's raw response (needs Tushare credentials)

Everything starts with two Tushare endpoints: `fut_basic` (the contract
catalogue per exchange, including exact delisting dates) and `fut_daily`
(daily bars per contract). The client below rate-limits (180 req/min) and
retries transient failures. Credentials may come from `TUSHARE_TOKEN` or the
private-config `tushare_token` key; without either this cell just skips.

In [ ]:
raw_daily = None
from sysdata.tushare.client import TushareClient
from sysdata.tushare.errors import TushareConfigError

try:
    client = TushareClient()
except TushareConfigError:
    client = None
    print("Tushare credentials or SDK unavailable - skipping the live vendor call")
else:
    raw_daily = client.fut_daily(ts_code="CU2609.SHF",
                                 start_date="20260601", end_date="20260610")
    display(raw_daily[["trade_date", "close", "settle", "vol", "oi"]])

Note the vendor gives both `close` (last trade) and `settle` (the exchange's
daily mark). **We store `close` as FINAL** — a backtest can realistically
fill near the close; nobody fills at the settlement average. A missing close
stays missing rather than being papered over with settle.

## Stage 1 — per-contract prices in parquet

The seed/update writes each contract as `OPEN/HIGH/LOW/FINAL/VOLUME` rows
stamped at the repo's notional close (23:00, naive), one parquet file per
contract per frequency. The comparison below does not rely on row order: it
turns the vendor's date strings into repository timestamps, joins on those
timestamps, and verifies that stored `FINAL` is exactly the traded `close`.

In [ ]:
from sysdata.data_blob import dataBlob
from sysproduction.data.prices import diagPrices
from sysobjects.contracts import futuresContract
from syscore.dateutils import DAILY_PRICE_FREQ

blob = dataBlob(log_name="notebook02")
price_store = diagPrices(blob).db_futures_contract_price_data
contract = futuresContract("SHFE_CU", "20260900")
stored = price_store.get_prices_at_frequency_for_contract_object(
    contract, frequency=DAILY_PRICE_FREQ
)
display(stored.tail())

if raw_daily is not None and not raw_daily.empty:
    vendor_close = raw_daily[["trade_date", "close"]].copy()
    vendor_close.index = (
        pd.to_datetime(vendor_close.pop("trade_date")) + pd.Timedelta(hours=23)
    )
    vendor_close = vendor_close.rename(columns={"close": "Tushare close"})
    close_check = vendor_close.join(
        stored[["FINAL"]].rename(columns={"FINAL": "stored FINAL"}),
        how="inner",
    ).sort_index()
    assert len(close_check) == len(vendor_close)
    assert np.allclose(
        close_check["Tushare close"], close_check["stored FINAL"],
        rtol=0.0, atol=0.0,
    )
    print(f"all {len(close_check)} overlapping closes match exactly")
    display(close_check)

## Stage 2 — contract state in MongoDB

Mongo holds the small mutable facts per contract: its **exact expiry** (the
vendor's delisting date, not an approximation) and whether it is currently
listed ("sampling").

In [ ]:
from sysproduction.data.contracts import dataContracts

contracts = dataContracts(blob)
chain = contracts.get_all_contract_objects_for_instrument_code("SHFE_CU")
db_contract = contracts.get_contract_from_db(contract)
print(f"SHFE_CU has {len(chain)} contracts in the database")
print(f"{contract.key}: expiry {db_contract.expiry_date}, "
      f"sampling={db_contract.currently_sampling}")

## Stage 3 — initialized multiple and adjusted series

Roll calendars turn contract prices into `multiple prices` (price + carry +
forward contracts side by side) and then into one back-adjusted series. The
calendar is an initialization/recovery artifact: after this build, production
reads current contract identities from the last multiple-price row.

In [ ]:
prices_stage = diagPrices(blob)
multiple = prices_stage.get_multiple_prices("SHFE_CU")
adjusted = prices_stage.get_adjusted_prices("SHFE_CU")
display(multiple.tail(3))

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
multiple["PRICE"].plot(ax=axes[0], title="SHFE_CU held-contract close")
axes[0].set_ylabel("price")
adjusted.plot(ax=axes[1], title="additive-Panama level (absolute level is arbitrary)")
axes[1].set_ylabel("adjusted price units")
axes[1].set_xlabel(f"date ({adjusted.index[0]:%Y-%m-%d} onward)")
plt.tight_layout()

## The daily update

```bash
python -m sysproduction.update_tushare_futures
```

does, in order: refresh the catalogue (6 requests) → upsert every contract's
expiry/sampling state into Mongo → for each *currently listed* contract,
re-fetch the last 7 days and append anything new → update CNHUSD. A typical
run summary looks like:

```
vendor=10919 internal=10929 written=872 ... rows=0 no_data=0 failures=0
```

(`rows=0` on a weekend: nothing new to append — the run is idempotent.)

After raw contracts update, the native daily multiple/adjusted process appends
the current `PRICE`, `FORWARD`, and `CARRY` contracts. It does not read the
calendar CSV. When a real contract roll is required,
`python -m sysproduction.interactive_update_roll_status` changes the
multiple-price identities and restitches adjusted prices; that is a separate
live-production decision.

**Designed to fail loudly** (see `docs/tushare_chinese_futures.md` for the
full failure model):

- If Tushare *changes already-stored history* inside the overlap window, the
  update refuses to write for that contract and names the changed dates.
  You accept new history deliberately:
  `python -m sysinit.futures.seed_price_data_from_tushare --instrument SHFE_CU --contract 202609 --no-resume`
- If Tushare *lists a brand-new product family*, known instruments still
  update but the run exits non-zero until you add a manifest row.
- Price spikes beyond `max_price_spike` block that contract's write.

The cell below runs the catalogue validation live (6 API calls) if the
client was configured — the same preflight the seed performs.

In [ ]:
import subprocess, sys

if client is not None:
    result = subprocess.run(
        [sys.executable, "-m", "sysinit.futures.seed_price_data_from_tushare",
         "--dry-run"],
        capture_output=True, text=True, cwd=R.REPO_ROOT,
    )
    if result.returncode != 0:
        diagnostic = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(
            "Tushare dry-run failed:\n" + diagnostic[-2_000:]
        )
    visible = [
        line
        for line in result.stdout.splitlines()
        if " DEBUG " not in line and " INFO " not in line
        and not line.startswith("Configuring sim logging")
    ]
    print("\n".join(visible[-12:]))
else:
    print("Tushare client unavailable - skipping the live catalogue check")

**Next**: notebook 02 — roll calendars, including the fully manual workflow.